In [1]:
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import factorial
from scipy.interpolate import RBFInterpolator
from scipy.optimize import root_scalar
from collections import defaultdict


In [2]:
# Apply Savitzky-Golay filter
from scipy.signal import savgol_filter
window_length = 11  # Must be odd
polyorder = 3

In [3]:
raw_df = pd.read_csv(
    "experiment_data.csv", 
    delimiter='\t', 
    names=['client_labda','server_mu', 'alpha', 'active_servers_num', 'run_iterations', 'request_id', 'server_processing_time_ideal', 
           'time_untill_next_client', 'sent_is_cpu','client_measured_time','server_processing_time_measured', 'is_cpu_received'], 
    header=None
)
raw_df.head()

,client_labda,server_mu,alpha,active_servers_num,run_iterations,request_id,server_processing_time_ideal,time_untill_next_client,sent_is_cpu,client_measured_time,server_processing_time_measured,is_cpu_received
0,0.3,1,0.1,1,500,0,0.617761,0.246800,0,623.0,0.618305,0.0
1,0.3,1,0.1,1,500,1,0.570009,0.125432,0,1054.0,0.570570,0.0
2,0.3,1,0.1,1,500,2,0.248522,0.005538,0,1288.0,0.248974,0.0
3,0.3,1,0.1,1,500,3,0.578565,0.505948,0,1358.0,0.579098,0.0
4,0.3,1,0.1,1,500,4,0.112288,0.157745,0,1298.0,0.112846,0.0


In [4]:
print(f"NO RESULT WAS SENT FOR - {sum(raw_df['sent_is_cpu']!=raw_df['is_cpu_received'])} requests")

NO RESULT WAS SENT FOR - 187 requests


In [5]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5078013 entries, 0 to 5078012
Data columns (total 12 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   client_labda                     float64
 1   server_mu                        int64  
 2   alpha                            float64
 3   active_servers_num               int64  
 4   run_iterations                   int64  
 5   request_id                       int64  
 6   server_processing_time_ideal     float64
 7   time_untill_next_client          float64
 8   sent_is_cpu                      int64  
 9   client_measured_time             float64
 10  server_processing_time_measured  float64
 11  is_cpu_received                  float64
dtypes: float64(7), int64(5)
memory usage: 464.9 MB


## Data cleaning

In [6]:
received_results_df = raw_df[raw_df['sent_is_cpu']==raw_df['is_cpu_received']].copy()

In [7]:
received_results_df = received_results_df[received_results_df['run_iterations'] == 500]

In [8]:
received_results_df.info()

<class 'pandas.DataFrame'>
Index: 5077826 entries, 0 to 5078012
Data columns (total 12 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   client_labda                     float64
 1   server_mu                        int64  
 2   alpha                            float64
 3   active_servers_num               int64  
 4   run_iterations                   int64  
 5   request_id                       int64  
 6   server_processing_time_ideal     float64
 7   time_untill_next_client          float64
 8   sent_is_cpu                      int64  
 9   client_measured_time             float64
 10  server_processing_time_measured  float64
 11  is_cpu_received                  float64
dtypes: float64(7), int64(5)
memory usage: 503.6 MB


## Convert to appropreate datatypes for mem-footprint reduction

In [9]:
# Convert categorical datatypes
received_results_df['client_labda'] = received_results_df['client_labda'].astype('category')
received_results_df['server_mu'] = received_results_df['server_mu'].astype('category')
received_results_df['alpha'] = received_results_df['alpha'].astype('category')

# Convert Integers datatypes
received_results_df['is_cpu_received'] = received_results_df['is_cpu_received'].astype('int64')

# Convert boolean datatypes
received_results_df['sent_is_cpu'] = received_results_df['sent_is_cpu'].astype('bool')

# Convert miliseconds to seconds
MILISECONDS_IN_SECOND = 1000
received_results_df['client_measured_time'] = received_results_df['client_measured_time'] / MILISECONDS_IN_SECOND

In [10]:
# The is_cpu flag must be the same sent_flat==recived_flag
assert(sum(received_results_df['sent_is_cpu'] != received_results_df['is_cpu_received']) == 0)

In [11]:
received_results_df = received_results_df.drop(['is_cpu_received'], axis=1)

In [12]:
# remove the problematic results with few examples
received_results_df = received_results_df[
    received_results_df['alpha'].isin({0.0, 0.2, 0.4, 0.6, 0.8, 1.0})
]

received_results_df = received_results_df[
    (received_results_df['active_servers_num'] > 20) & (received_results_df['active_servers_num'] < 694)
]

In [13]:
received_results_df.info()

<class 'pandas.DataFrame'>
Index: 4122481 entries, 100500 to 5078012
Data columns (total 11 columns):
 #   Column                           Dtype   
---  ------                           -----   
 0   client_labda                     category
 1   server_mu                        category
 2   alpha                            category
 3   active_servers_num               int64   
 4   run_iterations                   int64   
 5   request_id                       int64   
 6   server_processing_time_ideal     float64 
 7   time_untill_next_client          float64 
 8   sent_is_cpu                      bool    
 9   client_measured_time             float64 
 10  server_processing_time_measured  float64 
dtypes: bool(1), category(3), float64(4), int64(3)
memory usage: 267.3 MB


## Add experiment ID

In [14]:
run_id = 0
run_ids = []
prev_alpha = None
prev_n = None

for idx, row in received_results_df.iterrows():
    alpha = row['alpha']
    n = row['active_servers_num']
    req_id = row['request_id']
    
    if idx > 0:
        # Start a new run if parameters changed OR request_id resets to 0
        if (alpha != prev_alpha) or (n != prev_n) or (req_id == 0):
            run_id += 1
    
    run_ids.append(run_id)
    prev_alpha = alpha
    prev_n = n

received_results_df['run_id'] = run_ids

In [15]:
received_results_df.to_feather("received_results_df.feather")